In [ ]:
import json
from pathlib import Path
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

: 

##### Парсер файлов json

In [ ]:


DATA_DIR = Path("json_data")


def load_json_file(path: Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {path}")

    text = path.read_text(encoding="utf-8")
    payload = json.loads(text)
    if not isinstance(payload, list):
        raise ValueError(
            f"Expected a JSON array in {path}, got {type(payload).__name__}")

    records: list[dict] = []
    benchmark = path.stem
    for entry in payload:
        if not isinstance(entry, dict):
            continue
        row = entry.copy()
        row["source_file"] = path.name
        row["benchmark"] = benchmark
        if "columns" in row and "rows" in row:
            row["cells"] = int(row.get("columns", 0)) * int(row.get("rows", 0))
            row["aspect_ratio"] = row["columns"] / \
                row["rows"] if row["rows"] else np.nan
        records.append(row)
    return records


def load_json_data(path: Path) -> pd.DataFrame:
    path = Path(path)
    records: list[dict] = []
    if path.is_dir():
        for file in sorted(path.glob("*.json")):
            records.extend(load_json_file(file))
    else:
        records.extend(load_json_file(path))
    return pd.DataFrame(records)


df = load_json_data(DATA_DIR)
print("Loaded df:", df.shape)

In [ ]:
df

In [ ]:
new_df = df.drop(['columns', 'rows', 'seed', 'all_pairs_connected', 'source_file', 'cycle_count',
                  'memory_peak_bytes', 'benchmark', 'microseconds', 'longest_path'], axis=1).dropna()
new_df

In [ ]:
new_df.columns

In [ ]:
scaler = StandardScaler()
new_df_scaled = scaler.fit_transform(new_df)
print(
    f"Подготовлено данных: {new_df_scaled.shape[0]} объектов, {new_df_scaled.shape[1]} признаков.")

#### Временная разница между алгоритмами по генерации

In [ ]:
temp_df = df
# Создаем колонку в миллисекундах
temp_df['ms'] = temp_df['microseconds'] / 1000

sns.lineplot(data=temp_df, x="cells", y="ms", hue="benchmark", marker="o")
plt.title("Время выполнения в зависимости от количества ячеек")
plt.xlabel("Количество ячеек")
plt.legend(title="Алгоритм")
plt.ylabel("Время (миллисекунды)")

In [ ]:
pca = PCA(n_components=2)
principal_components = pca.fit_transform(new_df_scaled)
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])
df_pca['benchmark'] = df['benchmark'].values

### PC{number}
С помощью генерации библиотеки был создан взвешенный индекс, который берет какие-то параметры, добавляет веса и делает среднее. 
Например
`(W1*P1 + W2*P2 + W3*P3) / 3`
Так как это сжатие данных в меньшее пространство(векторы все дела)
То предоставляется возможность понять насколько много потеряно детальности
Процент считается сложными(!) математическими выражениями, основанные на собственных числах, матрицах ковариативности и подсчёт вклада(доли)

In [ ]:
print(
    f"Объясненная дисперсия: PC1 = {pca.explained_variance_ratio_[0]:.2%}, PC2 = {pca.explained_variance_ratio_[1]:.2%}")
print(f"Суммарно: {sum(pca.explained_variance_ratio_):.2%}")

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='benchmark',
                palette='bright', s=50, alpha=0.8)
plt.title('PCA: Проекция лабиринтов в пространство 2-х главных компонент (Глобальная структура)')
plt.xlabel(
    f'Главная компонента 1 ({pca.explained_variance_ratio_[0]:.1%} дисперсии)')
plt.ylabel(
    f'Главная компонента 2 ({pca.explained_variance_ratio_[1]:.1%} дисперсии)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Алгоритм')
plt.tight_layout()
plt.show()

In [ ]:
pc_weights = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=new_df.columns
)

print("\nИнтерпретация главных компонент (веса признаков):")
# Сортируем признаки для PC1 по абсолютному вкладу
print("\n--- Ведущие признаки для PC1 ---")
print(pc_weights['PC1'].abs().sort_values(ascending=False).head(5))
print("\n--- Ведущие признаки для PC2 ---")
print(pc_weights['PC2'].abs().sort_values(ascending=False).head(5))

### Промежуточный вывод

на графике явно видно, что у dfs огромные проблемы с ветвистостью, количеством тупиков, коридорностью.
Доминирующим по ветвисти является алгоритм ватсона,
По данным PC1(степень блокировок путей, возможности пойти в другую клетку) доминирующим является dfs, по PC2(степень ветвистости и выбора) является watson. Алгоритмы Watson, binary, prim, req_division являются очень схожими, хоть и отличимыми. Самым средним по показательям оказался алгоритм выращивания дерева.


# ПУНКТ 2


Composite Indicator (композитный индикатор) или индекс


In [ ]:
new_df

In [ ]:
factor_metric_for_dificult = {
    'branch_ratio': 1,
    'steps_count': 0,
    'wall_density': 1,
    'dead_end_count': 1,
    'dead_end_ratio': 1,
    'corridor_ratio': -1,
    'avg_degree': -1,
    'diameter': 1,
    'symmetry_score': -1,
    'fractal_dimension': 1,
    'aspl': 1,
    'entropy': 1,
    'cells': 1,
    'aspect_ratio': -1
}
# прямо ли коррелирует фактор с трудностью или обратно пропорционально.

#### Нормирование
Было проанализировано сразу 4 типа нормирования
min max scalling. Когда все значения внутри [0,1]
метод наименьших квадратов(zscore)
Метод на процентили(rank transform)


**Они ничем не отличаются в результате на данной выборке**, был оставлен zscrore

In [ ]:
df_zscore = new_df.apply(
    lambda x: (x-x.mean()) / x.std())
df_zscore = df_zscore.mul(pd.Series(factor_metric_for_dificult))
df_zscore.drop('steps_count', inplace=True, axis=1)
df_zscore

Данные отнормированы.

Подсчёт индексов методом равных весов. 
`(P1 + P2 + P3) /3`


In [ ]:
def create_index(df: pd.DataFrame) -> pd.DataFrame:
    df['index'] = df.sum(axis=1) / df.shape[1]
    return df

In [ ]:
df_zscore = create_index(df_zscore)

In [ ]:
def create_NCI_rank(new_df: pd.DataFrame) -> pd.DataFrame:
    new_df['benchmark'] = df['benchmark'].values
    median_new_df = new_df.groupby(
        'benchmark')['index'].median().reset_index()
    median_new_df.rename(columns={'index': 'median_index'}, inplace=True)
    median_new_df['NCI_rank'] = (rankdata(median_new_df['median_index'],
                                        method='average')
                                 / len(median_new_df)) \
                                        * 10
                                        
    median_new_df.sort_values('NCI_rank', inplace=True,ascending=False)
    median_new_df.drop(columns=['median_index'], inplace=True)
    median_new_df['NCI_rank'] = median_new_df['NCI_rank'].transform(func=round)
    return median_new_df


nci_df_zscore = create_NCI_rank(df_zscore)

In [ ]:
print("\nРейтинг NCI(Сложности прохождения) (на основе z-оценок и медиан):")
nci_df_zscore['benchmark'].index
nci_df_zscore